# 05 Evaluation and alternatives for DNS tunnel detection

[01 Discovery](01_discovery.ipynb) documents the labels and the two bursts.
[02 Baseline](02_baseline.ipynb) describes normal traffic and domain grouping.
[03 Features](03_features.ipynb) defines the features and stress tests.
[04 Results](04_results.ipynb) evaluates the Isolation Forest and groups alerts into incidents.

Run the cells in order from the repository root or the `notebooks` directory, using the
environment defined in `requirements.txt`. The notebook reads the local `dns.log` and
`data/labels.csv` through `src.prepare` and runs offline.

The dataset is a single capture with two related bursts. Notebook 01 investigated the
second burst as a replay. Fitting and calibration use data from before the test cutoff.
Notebook 04 already examined the test period, so the comparisons here are retrospective.
Confirming a configuration requires a new capture.

The main comparison uses the project's `forced` feature set at 60 s with `folds_median`,
at 300 s with `folds_median`, and at 300 s with `val_max`. The main comparison uses seeds
0–9. The training and stress experiments use seeds 0–4. Comparisons pair runs by seed.
Standard deviations show variation across seeds within this capture. PR AUC is reported
as average precision. A query raises an alert when its score exceeds the threshold.

In [ ]:
import hashlib
import platform
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "prepare.py").is_file()
)
sys.path.insert(0, str(ROOT))

from src.baselines import run_rule
from src.detector import DEFAULTS, anomaly_score, fit_forest, run_detector, split_validation
from src.evaluation import confusion
from src.features import META_COLUMNS, build_features, drift_table, feature_sets
from src.incidents import group_alerts, incident_metrics
from src.prepare import DEFAULT_CUT, _find_raw_dir, derive_columns, load_raw, prepare
from src.variants import add_benign_clients, inject_repeats, slow_tunnel

SEEDS = tuple(range(10))
STRESS_SEEDS = SEEDS[:5]
CONFIGS = {
    "60 s, folds_median": (60, "folds_median"),
    "300 s, folds_median": (300, "folds_median"),
    "300 s, val_max": (300, "val_max"),
}
BASELINE = next(iter(CONFIGS))
WINDOWS = (60, 300)
FEATURES = {window: feature_sets(window)["forced"] for window in WINDOWS}
raw = load_raw()
prep = prepare(raw, session="second")
matrices = {window: build_features(prep, window) for window in WINDOWS}

## 1. Data Leakage and chronological splits

A feature at time t should depend only on the current row and earlier rows for the same
client and parent domain. Adding later traffic should leave earlier feature values unchanged.

Check the labels and chronological split boundaries. Rebuild features on the rows before
the validation and test boundaries and compare them with the corresponding rows from
the full run. A small fixture checks the window boundary, timestamp ties, and isolation
between clients and domains. Timestamp ties follow file order. The rolling window is
(t − window, t]. Cumulative features use the history of the group up to the current query.

In [ ]:
split_checks = []
for window, matrix in matrices.items():
    train = matrix.loc[matrix.split.eq("train")]
    test = matrix.loc[matrix.split.eq("test")]
    fit, val = split_validation(train)
    assert prep.ts.is_monotonic_increasing and prep.row_id.is_unique
    assert train.y.eq(0).all() and test.y.nunique() == 2
    assert fit.ts.max() < val.ts.min() <= val.ts.max() < test.ts.min()
    for boundary in (val.ts.min(), test.ts.min()):
        prefix = prep.loc[prep.ts.lt(boundary)]
        pd.testing.assert_frame_equal(
            build_features(prefix, window), matrix.loc[matrix.ts.lt(boundary)]
        )

    # Reuse prepared rows and replace only fields read by the context feature calculation.
    fixture = prep.head(6).copy().assign(
        ts=[0, 1, 1, window, window, window],
        client=["a", "a", "b", "a", "a", "a"],
        parent=["x.test", "x.test", "x.test", "x.test", "y.test", "x.test"],
        name=["a.x.test", "b.x.test", "c.x.test", "c.x.test", "d.y.test", "b.x.test"],
    )
    observed = build_features(fixture, window)
    np.testing.assert_allclose(observed[f"uniq_names_{window}s"], [1, 2, 1, 2, 1, 2])
    np.testing.assert_allclose(observed[f"uniq_ratio_{window}s"], [1, 1, 1, 1, 1, 2/3])
    np.testing.assert_allclose(observed["norepeat_tol"], np.log1p([1, 2, 1, 3, 1, 3])
                               * np.array([1, 1, 1, 1, 1, 3/4]))
    split_checks.append({"window_s": window, "fit": len(fit), "validation": len(val),
                         "test": len(test), "test_tunnel": int(test.y.sum())})

display(pd.DataFrame(split_checks).set_index("window_s"))
print("Both windows passed the prefix, boundary and grouping checks.")

,fit,validation,test,test_tunnel
window_s,,,,
60,1274,318,1831,222
300,1274,318,1831,222


Both windows passed the prefix, boundary and grouping checks.


### Summary: Data Leakage and Chronological Split Validation

The experiment validates the chronological data split and rolling feature calculations for 60-second and 300-second observation windows.

Both configurations use 1,274 training queries, 318 validation queries, and 1,831 test queries, including 222 tunnel queries.

All prefix, boundary, and grouping checks passed for both window sizes.

These results support the correctness of the chronological feature calculations and reduce the risk of future-information leakage.

However, the checks do not independently establish that every part of the model training and threshold calibration pipeline is leakage-free.

A validation or test query can use DNS traffic already observed, including traffic from
the preceding partition. That context is available when scoring a live stream. A random
row split could put queries with overlapping rolling windows in both fitting and validation
sets. Section 3 checks what happens when one window separates the two sets. The buffer
separates the rolling windows, while cumulative features still use earlier observations.

## 2. Parent domains and the Public Suffix List

Taking the last two labels groups distinct registrations under `co.uk` together.
The project uses the offline ICANN snapshot bundled with `tldextract` to identify the
registered domain. For an unknown suffix, the project takes the last two labels. It uses
the whole name for a single-label host. The [Public Suffix List](https://publicsuffix.org/list/)
documents the suffixes outside the control of individual registrants.

Test the project's extraction function with `co.uk`, mixed case, a trailing dot,
an unknown suffix and a single-label host. Then count how many queries in the capture
would receive a different parent under the last-two-labels rule. The examples use
reserved names and can be evaluated offline.

In [ ]:
def last_two(name):
    return ".".join(name.lower().rstrip(".").split(".")[-2:])

examples = {
    "cdn.Example.CO.UK.": "example.co.uk",
    "api.other.co.uk": "other.co.uk",
    "example.com": "example.com",
    "local.lab.invalid": "lab.invalid",
    "LOCALHOST": "localhost",
}
fixture = raw.head(len(examples)).assign(query=list(examples), label="benign")
parents = derive_columns(fixture)[["query", "parent", "subdomain"]]
parents["last_two_labels"] = parents["query"].map(last_two)
assert parents.parent.tolist() == list(examples.values())
assert parents.subdomain.tolist() == ["cdn", "api", "", "local", ""]
assert parents.last_two_labels.iloc[:2].nunique() == 1
assert parents.parent.iloc[:2].nunique() == 2

display(parents)

named = prep.assign(last_two_labels=prep.name.map(last_two))
changed = named.loc[named.parent.ne(named.last_two_labels)]

display(changed.groupby(["last_two_labels", "parent"]).size().rename("queries").to_frame())
print(f"The last-two-labels rule merges the two co.uk examples. "
      f"In the capture it assigns a different parent to {len(changed)} of {len(named)} queries.")

,query,parent,subdomain,last_two_labels
0,cdn.Example.CO.UK.,example.co.uk,cdn,co.uk
1,api.other.co.uk,other.co.uk,api,co.uk
2,example.com,example.com,,example.com
3,local.lab.invalid,lab.invalid,local,lab.invalid
4,LOCALHOST,localhost,,localhost


queries
last_two_labels parent                     
co.uk           guardianapps.co.uk       12
                guim.co.uk               39
                ophan.co.uk              12
in-addr.arpa    172.in-addr.arpa          2
                192.in-addr.arpa          3

The last-two-labels rule merges the two co.uk examples. In the capture it assigns a different parent to 68 of 3423 queries.


### Summary: Parent-Domain Extraction and Public Suffix List

The experiment demonstrates why correct registrable-domain extraction is important for DNS anomaly detection.

A simple last-two-labels approach incorrectly groups different `.co.uk` domains under the same parent.

In the dataset, 63 queries belong to three distinct `.co.uk` domains. Another five reverse-DNS queries show differences in grouping under `in-addr.arpa`.

Incorrect grouping can distort rolling features such as the number of unique DNS names per client and parent domain.

The tested extraction method correctly handles the provided examples.

Notebooks 03 and 04 call the same PSL-based extraction through `make_matrix()` and `prepare()`. Notebook 05 calls `prepare()` directly. The reported main experiments already use this parent-domain grouping.

## 3. Threshold calibration and fold stability

The threshold can vary between busy and quiet periods. A 300 s feature window could
change alert volume and recall even if the scores rank queries similarly.

Pass only pre-cutoff rows to the training API. Check that `val_max` raises zero alerts
on the final 20% of those rows, since an alert requires a score strictly above the
threshold. Reuse the same fitted forest for both thresholds at each window and seed.
Compare fold thresholds within each seed.

For `folds_median`, each fold holds out one contiguous block and fits on the other
four. An early block can therefore be scored by a model fitted on later normal traffic.
All five blocks precede the test cutoff. The forward check evaluates blocks 2–5 using
only earlier fitting rows, with a one-window gap before each validation block. The
forward check measures calibration sensitivity. The main comparison uses the three
settings specified above.

In [ ]:
def calibrate(matrix, columns, seed):
    train = matrix.loc[matrix.split.eq("train")]
    assert train.y.eq(0).all()
    assert not set(columns).intersection(META_COLUMNS)
    scored, info = run_detector(train, columns, random_state=seed)
    val = scored.loc[scored.role.eq("val"), "score"]
    assert len(val) and info["candidates"]["val_max"] == val.max()
    assert not scored.loc[scored.role.eq("val"), "alert"].any()
    assert info["candidates"]["folds_median"] == info["folds"]["line"].median()
    return info


def score_test(matrix, info, method):
    test = matrix.loc[matrix.split.eq("test")].copy()
    test["role"] = "test"
    test["score"] = anomaly_score(info["model"], test[info["features"]])
    test["alert"] = test.score.gt(info["candidates"][method]).astype(int)
    assert np.isfinite(test.score).all()
    return test


def metrics(test):
    return {
        "roc_auc": roc_auc_score(test.y, test.score),
        "pr_auc": average_precision_score(test.y, test.score),
        **confusion(test.y, test.alert),
        **incident_metrics(test),
    }


infos = {
    (window, seed): calibrate(matrices[window], FEATURES[window], seed)
    for window in WINDOWS for seed in SEEDS
}
scored = {
    (config, seed): score_test(matrices[window], infos[window, seed], method)
    for config, (window, method) in CONFIGS.items() for seed in SEEDS
}
results = pd.DataFrame([
    {"config": config, "seed": seed, **metrics(test)}
    for (config, seed), test in scored.items()
])
headline = ["roc_auc", "pr_auc", "recall", "precision", "fp", "fpr"]
summary = results.groupby("config", sort=False)[headline].agg(["mean", "std"])

display(summary.round(4))
paired = results.pivot(index="seed", columns="config", values=["recall", "fp", "precision"])
delta = {
    config: paired.xs(config, level="config", axis=1)
    - paired.xs(BASELINE, level="config", axis=1)
    for config in CONFIGS if config != BASELINE
}

display(pd.concat(delta, names=["config"]).groupby("config", sort=False).agg(["mean", "min", "max"]).round(4))

roc_auc          pr_auc          recall         precision  \
                       mean     std    mean     std    mean     std      mean   
config                                                                          
60 s, folds_median   0.9957  0.0002  0.9931  0.0004  0.9869  0.0014    0.7442   
300 s, folds_median  0.9959  0.0002  0.9934  0.0004  0.9865  0.0000    0.9609   
300 s, val_max       0.9959  0.0002  0.9934  0.0004  0.9865  0.0000    0.9853   

                               fp             fpr          
                        std  mean     std    mean     std  
config                                                     
60 s, folds_median   0.0238  75.6  9.6862  0.0470  0.0060  
300 s, folds_median  0.0197   9.0  4.7610  0.0056  0.0030  
300 s, val_max       0.0118   3.3  2.7101  0.0021  0.0017

recall                 fp             precision          \
                       mean     min  max  mean   min   max      mean     min   
config                                                                         
300 s, folds_median -0.0005 -0.0045  0.0 -66.6 -77.0 -57.0    0.2167  0.1952   
300 s, val_max      -0.0005 -0.0045  0.0 -72.3 -96.0 -61.0    0.2411  0.2063   

                             
                        max  
config                       
300 s, folds_median  0.2409  
300 s, val_max       0.3024

### Summary: Threshold Calibration and Window Comparison

The experiment compares three Isolation Forest configurations across ten random seeds.

Increasing the observation window from 60 to 300 seconds substantially reduces false positives while maintaining almost identical recall.

The original 60-second configuration produces an average of 75.6 false positives, with a precision of 0.7442 and recall of 0.9869.

Using a 300-second window with `folds_median` reduces false positives to 9.0 and increases precision to 0.9609.

The `val_max` threshold with a 300-second window performs even better, producing only 3.3 false positives and achieving a precision of 0.9853, while maintaining recall at 0.9865.

Compared with the baseline, this represents a 95.6% reduction in false positives, with a negligible decrease in recall.

The results demonstrate that observation window size and threshold calibration strongly influence operational detection performance.

However, the improved configuration should be considered exploratory until its performance has been validated without test-set-based selection.

In [ ]:
fold_rows = []
for (window, seed), info in infos.items():
    train = matrices[window].loc[lambda d: d.split.eq("train")]
    blocks = np.array_split(np.arange(len(train)), 5)
    for idx, old in zip(blocks, info["folds"].itertuples(index=False)):
        held = train.iloc[idx]
        assert held.ts.min() == old.start and held.ts.max() == old.end
        fit = train.loc[train.ts.le(old.start - window)]
        forward = np.nan
        if old.fold > 1:
            assert len(fit) and fit.ts.max() <= held.ts.min() - window
            model = fit_forest(fit[FEATURES[window]], random_state=seed)
            forward = anomaly_score(model, held[FEATURES[window]]).max()
        fold_rows.append({"window_s": window, "seed": seed, "fold": old.fold,
                          "retrospective": old.line, "forward": forward, "forward_fit_rows": len(fit)})

fold_results = pd.DataFrame(fold_rows)

display(fold_results.groupby(["window_s", "fold"])[
    ["retrospective", "forward", "forward_fit_rows"]
].mean().round(4))
same_blocks = fold_results.loc[fold_results.fold.gt(1)]
spread = same_blocks.groupby(["window_s", "seed"])[["retrospective", "forward"]].agg(np.ptp)

display(spread.groupby("window_s").agg(["mean", "min", "max"]).round(4))
for config, effect in delta.items():
    print(f"{config} versus {BASELINE}: mean change of {effect.fp.mean():+.1f} false positives "
          f"and {100 * effect.recall.mean():+.2f} percentage points in recall.")

print("The spread uses the same four validation blocks. "
      "The fitted models differ in how much earlier traffic they have seen.")

retrospective  forward  forward_fit_rows
window_s fold                                          
60       1            0.7142      NaN               0.0
         2            0.6795   0.7033             305.0
         3            0.6725   0.6277             413.0
         4            0.6111   0.6168             546.0
         5            0.6749   0.6607             811.0
300      1            0.7308      NaN               0.0
         2            0.6512   0.6748             217.0
         3            0.6850   0.6794             258.0
         4            0.6212   0.6721             300.0
         5            0.6918   0.6813             305.0

retrospective                 forward                
                  mean     min     max    mean     min     max
window_s                                                      
60              0.0707  0.0546  0.0814  0.0875  0.0698  0.1021
300             0.0710  0.0499  0.0862  0.0123  0.0057  0.0202

300 s, folds_median versus 60 s, folds_median: mean change of -66.6 false positives and -0.05 percentage points in recall.
300 s, val_max versus 60 s, folds_median: mean change of -72.3 false positives and -0.05 percentage points in recall.
The spread uses the same four validation blocks. The fitted models differ in how much earlier traffic they have seen.


### Summary: Retrospective and Forward Threshold Validation

This experiment compares retrospective and forward threshold calibration for 60-second and 300-second observation windows.

Forward validation trains each model exclusively on traffic preceding the corresponding validation block, providing a more realistic approximation of deployment conditions.

The 300-second configuration shows substantially more stable forward-calibrated thresholds, ranging from 0.6721 to 0.6813, compared with 0.6168 to 0.7033 for the 60-second configuration.

The reported mean threshold spread decreases from 0.0875 for 60 seconds to 0.0123 for 300 seconds.

However, the forward models use different amounts of preceding training data, and the reported results do not establish the test performance of forward-calibrated thresholds.

The experiment therefore supports the stability of the 300-second calibration approach, but does not yet provide independent confirmation of its operational detection performance.

The two 300 s configurations use identical scores, so the threshold accounts for any
change in alert volume or recall between them. By definition, a fold maximum produces
zero alerts on that fold's validation scores. The median of five maxima can still produce
alerts on validation data scored by the final model.

## 4. Representativeness of default and strict training

A short, quiet training period should cover less of the later normal traffic. Start with
the original `first` and `second` setups, then compare the two training cutoffs
on the same `second` test period. In the second comparison, the detector only fits
and calibrates on rows before the strict cutoff. Traffic between the cutoffs remains
available as prior feature history. This keeps the test features identical.

Check that all fitting and calibration rows are benign. Compare training size, domain
coverage and the share of benign test rows outside the training feature range. For the
comparison on a common test set, verify that row IDs, labels and features match exactly.
The original `first` and `second` setups cover different test traffic and bursts,
so their results are descriptive.

In [ ]:
window, method = CONFIGS[BASELINE]
default = matrices[window]
strict_original = build_features(prepare(raw, session="first"), window)
early_end = DEFAULT_CUT["first"].tz_localize("UTC").timestamp()
strict_same_test = default.loc[default.ts.lt(early_end) | default.split.eq("test")].copy()
case_matrices = {
    "default, second": default,
    "strict, first": strict_original,
    "strict-cut, same second test": strict_same_test,
}
pd.testing.assert_frame_equal(
    default.loc[default.split.eq("test")], strict_same_test.loc[strict_same_test.split.eq("test")]
)
coverage = []
strict_rows = []
for case, matrix in case_matrices.items():
    train = matrix.loc[matrix.split.eq("train")]
    normal = matrix.loc[matrix.split.eq("test") & matrix.y.eq(0)]
    assert train.y.eq(0).all()
    coverage.append({"setup": case, "training_rows": len(train), "training_domains": train.parent.nunique(),
                     "benign_test_rows": len(normal), "share_benign_queries_on_unseen_domains":
                     (~normal.parent.isin(train.parent)).mean()})
    
    display(case, drift_table(matrix, FEATURES[window]))
    for seed in STRESS_SEEDS:
        info = infos[window, seed] if case == "default, second" else calibrate(matrix, FEATURES[window], seed)
        strict_rows.append({"setup": case, "seed": seed, **metrics(score_test(matrix, info, method))})

display(pd.DataFrame(coverage).set_index("setup").round(4))
strict_results = pd.DataFrame(strict_rows)
strict_summary = strict_results.groupby("setup", sort=False)[
    ["pr_auc", "recall", "fp", "fpr", "false_incidents_per_hour"]
].mean()

display(strict_summary.round(4))
rates = strict_summary["fpr"]

print(f"On the same second-session test period, using the strict cutoff moves the FPR "
      f"from {rates.iloc[0]:.2%} to {rates.iloc[2]:.2%}. "
      "This cuts training volume and time coverage and changes the mix of traffic.")

'default, second'

,train max,test benign max,test benign above train max,tunnel median
name_bytes,103.000,93.000,0.000,19.000
uniq_names_60s,16.000,28.000,0.032,111.500
uniq_ratio_60s,1.000,1.000,0.000,1.000
norepeat_tol,1.811,2.047,0.005,4.723


'strict, first'

,train max,test benign max,test benign above train max,tunnel median
name_bytes,63.000,103.000,0.005,19.000
uniq_names_60s,4.000,28.000,0.171,111.500
uniq_ratio_60s,1.000,1.000,0.000,1.000
norepeat_tol,1.609,2.047,0.004,4.723


'strict-cut, same second test'

,train max,test benign max,test benign above train max,tunnel median
name_bytes,63.000,93.000,0.006,19.000
uniq_names_60s,4.000,28.000,0.213,111.500
uniq_ratio_60s,1.000,1.000,0.000,1.000
norepeat_tol,1.609,2.047,0.005,4.723


,training_rows,training_domains,benign_test_rows,share_benign_queries_on_unseen_domains
setup,,,,
"default, second",1592,172,1609,0.3089
"strict, first",244,34,2957,0.7423
"strict-cut, same second test",244,34,1609,0.7601


,pr_auc,recall,fp,fpr,false_incidents_per_hour
setup,,,,,
"default, second",0.9931,0.9874,77.0,0.0479,6.0148
"strict, first",0.9832,0.9865,669.2,0.2263,41.7290
"strict-cut, same second test",0.9850,0.9865,416.8,0.2590,31.8785


On the same second-session test period, using the strict cutoff moves the FPR from 4.79% to 25.90%. This cuts training volume and time coverage and changes the mix of traffic.


### Summary: Training Representativeness and Distribution Shift

This experiment evaluates how the amount and diversity of benign training data affect Isolation Forest performance.

The default configuration uses 1,592 training queries covering 172 parent domains. The strict configuration uses only 244 queries covering 34 domains.

When evaluated on the same second-session test period, reducing the training coverage increases the false-positive rate from 4.79% to 25.90%, while recall remains approximately 98.7%.

The reduced training coverage also increases the proportion of benign test queries exceeding the observed training maximum for `uniq_names_60s`, from 3.2% to 21.3%.

Furthermore, the proportion of benign queries involving previously unseen domains increases from 30.89% to 76.01%.

These findings demonstrate that the detector is sensitive to training-data representativeness. A model trained on a narrow sample of normal DNS behavior may incorrectly flag legitimate activity as anomalous.

The results support the need for representative benign training data, careful calibration, and monitoring for distribution shift in operational deployments.

However, the experiment does not isolate training volume, temporal coverage, and domain diversity as independent causal factors.

## 5. Offline stress tests on DNS telemetry

A lower query rate can change rolling features. Repeated names should suppress
`norepeat_evidence` more strongly than `norepeat_tol`. More clients can generate
more false alerts even if the false-positive rate stays close to the reference rate.

Compare the reference with the two slow-tunnel scenarios from notebook 04, a single
repeated name, and ten benign hosts. Nine of those hosts are simulated from existing
benign test rows. Copy those rows without shifting timestamps so the test duration
stays fixed in the host experiment. Use the original fitted models and thresholds.
Check that the training features are identical and that each simulation changes the
expected rows. All changes are made to the recorded data in memory.

In [ ]:
train_prep = prep.loc[prep.split.eq("train")]
test_prep = prep.loc[prep.split.eq("test")]
repeat = inject_repeats(prep, n=1)
noisy_test = add_benign_clients(test_prep, n_extra=9, max_shift_s=0, seed=0)
noisy = pd.concat([train_prep, noisy_test], ignore_index=True)
stress_cases = {
    "reference": prep,
    "20x slower": slow_tunnel(prep, k=20),
    "60x slower": slow_tunnel(prep, k=60),
    "one repeated name": repeat,
    "10 benign hosts": noisy,
}
tunnel_prep = prep.loc[prep.y.eq(1)]

for scale, case in [(20, "20x slower"), (60, "60x slower")]:
    slower = stress_cases[case]
    pd.testing.assert_frame_equal(
        slower.loc[slower.y.eq(0)].reset_index(drop=True),
        prep.loc[prep.y.eq(0)].reset_index(drop=True),
    )
    tunnel = slower.loc[slower.y.eq(1)]
    pd.testing.assert_frame_equal(tunnel[["row_id", "name"]].reset_index(drop=True),
                                  tunnel_prep[["row_id", "name"]].reset_index(drop=True))
    np.testing.assert_allclose(np.diff(tunnel.ts), scale * np.diff(tunnel_prep.ts), atol=1e-6)

extra = repeat.loc[repeat.row_id.lt(0)]
assert len(extra) == 1 and extra.y.eq(1).all() and extra.split.eq("test").all()
assert extra.name.iloc[0] in set(tunnel_prep.name)
assert noisy_test.split.eq("test").all()
assert noisy_test.loc[noisy_test.y.eq(0), "client"].nunique() == 10
assert noisy_test.y.sum() == test_prep.y.sum()
assert noisy_test.ts.agg(["min", "max"]).equals(test_prep.ts.agg(["min", "max"]))

stress_rows = []
stress_scored = {}

for case, prepared in stress_cases.items():
    assert prepared.row_id.is_unique
    case_windows = {window: build_features(prepared, window) for window in WINDOWS}
    for window, frame in case_windows.items():
        pd.testing.assert_frame_equal(
            frame.loc[frame.split.eq("train")], matrices[window].loc[lambda d: d.split.eq("train")]
        )
    for config, (window, method) in CONFIGS.items():
        for seed in STRESS_SEEDS:
            test = score_test(case_windows[window], infos[window, seed], method)
            stress_scored[case, config, seed] = test
            stress_rows.append({"scenario": case, "config": config, "seed": seed,
                                "test_duration_min": (test.ts.max() - test.ts.min()) / 60,
                                "positive_share": test.y.mean(), **metrics(test)})

repeat_matrix = build_features(repeat, 60)
repeat_features = repeat_matrix.loc[repeat_matrix.row_id.eq(extra.row_id.iloc[0])]
assert repeat_features.norepeat_evidence.eq(0).all() and repeat_features.norepeat_tol.gt(0).all()

display(repeat_features[["uniq_names_60s", "norepeat_evidence", "norepeat_tol"]].round(4))
stress_results = pd.DataFrame(stress_rows)
stress_summary = stress_results.groupby(["scenario", "config"], sort=False)[
    ["test_duration_min", "positive_share", "pr_auc", "recall", "fp", "fpr", "false_incidents"]
].mean()

display(stress_summary.round(4))
reference, noisy_result = (stress_summary.loc[(case, BASELINE)]
                           for case in ("reference", "10 benign hosts"))

print(f"With ten hosts, the baseline averages {noisy_result.fp:.1f} false positives "
      f"versus {reference.fp:.1f} on the reference traffic. "
      f"The FPR moves from {reference.fpr:.2%} to {noisy_result.fpr:.2%}. "
      "At the inserted repeat, norepeat_tol stays positive and norepeat_evidence drops to zero.")

,uniq_names_60s,norepeat_evidence,norepeat_tol
2357,31.0,0.0,3.3574


test_duration_min  positive_share  \
scenario          config                                                   
reference         60 s, folds_median             39.9015          0.1212   
                  300 s, folds_median            39.9015          0.1212   
                  300 s, val_max                 39.9015          0.1212   
20x slower        60 s, folds_median             39.9015          0.1212   
                  300 s, folds_median            39.9015          0.1212   
                  300 s, val_max                 39.9015          0.1212   
60x slower        60 s, folds_median             45.3686          0.1212   
                  300 s, folds_median            45.3686          0.1212   
                  300 s, val_max                 45.3686          0.1212   
one repeated name 60 s, folds_median             39.9015          0.1217   
                  300 s, folds_median            39.9015          0.1217   
                  300 s, val_max                 39.9015          0.1217   
10 benign hosts   60 s, folds_median             39.9015          0.0136   
                  300 s, folds_median            39.9015          0.0136   
                  300 s, val_max                 39.9015          0.0136   

                                       pr_auc  recall     fp     fpr  \
scenario          config                                               
reference         60 s, folds_median   0.9931  0.9874   77.0  0.0479   
                  300 s, folds_median  0.9932  0.9865   10.6  0.0066   
                  300 s, val_max       0.9932  0.9865    3.6  0.0022   
20x slower        60 s, folds_median   0.9895  0.9829   77.0  0.0479   
                  300 s, folds_median  0.9932  0.9865   10.6  0.0066   
                  300 s, val_max       0.9932  0.9865    3.6  0.0022   
60x slower        60 s, folds_median   0.9891  0.9829   77.0  0.0479   
                  300 s, folds_median  0.9930  0.9847   10.6  0.0066   
                  300 s, val_max       0.9930  0.9829    3.6  0.0022   
one repeated name 60 s, folds_median   0.9932  0.9874   77.0  0.0479   
                  300 s, folds_median  0.9933  0.9865   10.6  0.0066   
                  300 s, val_max       0.9933  0.9865    3.6  0.0022   
10 benign hosts   60 s, folds_median   0.9875  0.9874  720.0  0.0447   
                  300 s, folds_median  0.9884  0.9865   78.6  0.0049   
                  300 s, val_max       0.9884  0.9865   33.2  0.0021   

                                       false_incidents  
scenario          config                                
reference         60 s, folds_median               4.0  
                  300 s, folds_median              4.4  
                  300 s, val_max                   1.8  
20x slower        60 s, folds_median               4.0  
                  300 s, folds_median              4.4  
                  300 s, val_max                   1.8  
60x slower        60 s, folds_median               4.0  
                  300 s, folds_median              4.4  
                  300 s, val_max                   1.8  
one repeated name 60 s, folds_median               4.0  
                  300 s, folds_median              4.4  
                  300 s, val_max                   1.8  
10 benign hosts   60 s, folds_median              41.6  
                  300 s, folds_median             22.0  
                  300 s, val_max                   9.6

With ten hosts, the baseline averages 720.0 false positives versus 77.0 on the reference traffic. The FPR moves from 4.79% to 4.47%. At the inserted repeat, norepeat_tol stays positive and norepeat_evidence drops to zero.


### Summary: Robustness and Evasion Testing

The robustness experiments evaluate the Isolation Forest detector under slower DNS tunneling, repeated query names, and increased benign traffic volume.

The detector maintains high recall under the tested slowdown scenarios. Even with a 60-fold slowdown, recall remains approximately 98.3%, and PR AUC remains above 0.989.

Repeating one DNS query name has negligible impact on detection performance.

However, increasing benign traffic to ten hosts substantially increases the absolute false-alert burden. The original 60-second baseline produces approximately 720 false-positive queries and 41.6 false incidents, compared with 33.2 false-positive queries and 9.6 false incidents for the 300-second `val_max` configuration.

These findings show that the detector is relatively robust under the tested conditions, while operational alert volume remains an important limitation.

The simulations do not establish resistance to adaptive attackers, realistic long-duration low-and-slow tunneling, or independent host populations.

The simulated hosts share traffic from one capture, so their activity is correlated.
The slower tunnel scenarios can extend beyond the last recorded benign query. The
table reports test duration and the share of tunnel queries. Adding benign hosts
changes that share, which can affect average precision.

## 6. What detection delay measures

Detection can take longer in seconds even if the detector sees roughly the same
number of tunnel queries before raising its first alert.

Measure from the first labelled tunnel query to the first alert on a labelled tunnel
query. Record how many tunnel queries came before that alert and the phase of the
first hit. This is elapsed time in the capture. It excludes network latency, processing
time and analyst triage. A missed burst counts against the detection rate and has an
undefined delay. Ties follow the original row order. Each scenario contains one burst.

In [ ]:
def delay(test):
    tunnel = test.loc[test.y.eq(1)].sort_values("ts", kind="stable")
    hits = np.flatnonzero(tunnel.alert.to_numpy())
    if not len(hits):
        return {"detected": False, "delay_s": np.nan, "queries_before_first_alert": np.nan, "phase": None}
    first = tunnel.iloc[hits[0]]
    return {"detected": True, "delay_s": first.ts - tunnel.ts.iloc[0],
            "queries_before_first_alert": hits[0], "phase": first.phase}


delay_rows = []
for (case, config, seed), test in stress_scored.items():
    measured = delay(test)
    expected = incident_metrics(test)["seconds_to_first_alert"]
    np.testing.assert_allclose(measured["delay_s"], expected, equal_nan=True)
    delay_rows.append({"scenario": case, "config": config, "seed": seed, **measured})
delay_results = pd.DataFrame(delay_rows)
delay_summary = delay_results.groupby(["scenario", "config"], sort=False).agg(
    runs=("detected", "size"), detected=("detected", "sum"),
    median_delay_s=("delay_s", "median"), max_delay_s=("delay_s", "max"),
    median_queries_before_first_alert=("queries_before_first_alert", "median"),
)

display(delay_summary.round(3))

display(delay_results.loc[delay_results.seed.eq(0), ["scenario", "config", "phase"]])

for case in ("reference", "20x slower", "60x slower"):
    row = delay_summary.loc[case, BASELINE]
    print(f"{case}, baseline: detected in {row.detected:.0f}/{row.runs:.0f} runs. "
          f"Median delay: {row.median_delay_s:.3f} s, "
          f"with {row.median_queries_before_first_alert:g} tunnel queries before the first alert.")

print("Report the time delay alongside the number of queries before the first alert and the number of missed runs.")

runs  detected  median_delay_s  \
scenario          config                                                
reference         60 s, folds_median      5         5           0.007   
                  300 s, folds_median     5         5           0.007   
                  300 s, val_max          5         5           0.007   
20x slower        60 s, folds_median      5         5           0.146   
                  300 s, folds_median     5         5           0.146   
                  300 s, val_max          5         5           0.146   
60x slower        60 s, folds_median      5         5           0.439   
                  300 s, folds_median     5         5           0.439   
                  300 s, val_max          5         5           0.439   
one repeated name 60 s, folds_median      5         5           0.007   
                  300 s, folds_median     5         5           0.007   
                  300 s, val_max          5         5           0.007   
10 benign hosts   60 s, folds_median      5         5           0.007   
                  300 s, folds_median     5         5           0.007   
                  300 s, val_max          5         5           0.007   

                                       max_delay_s  \
scenario          config                             
reference         60 s, folds_median         0.007   
                  300 s, folds_median        0.007   
                  300 s, val_max             0.007   
20x slower        60 s, folds_median         0.146   
                  300 s, folds_median        0.146   
                  300 s, val_max             0.146   
60x slower        60 s, folds_median         0.439   
                  300 s, folds_median        0.439   
                  300 s, val_max             0.439   
one repeated name 60 s, folds_median         0.007   
                  300 s, folds_median        0.007   
                  300 s, val_max             0.007   
10 benign hosts   60 s, folds_median         0.007   
                  300 s, folds_median        0.007   
                  300 s, val_max             0.007   

                                       median_queries_before_first_alert  
scenario          config                                                  
reference         60 s, folds_median                                 3.0  
                  300 s, folds_median                                3.0  
                  300 s, val_max                                     3.0  
20x slower        60 s, folds_median                                 3.0  
                  300 s, folds_median                                3.0  
                  300 s, val_max                                     3.0  
60x slower        60 s, folds_median                                 3.0  
                  300 s, folds_median                                3.0  
                  300 s, val_max                                     3.0  
one repeated name 60 s, folds_median                                 3.0  
                  300 s, folds_median                                3.0  
                  300 s, val_max                                     3.0  
10 benign hosts   60 s, folds_median                                 3.0  
                  300 s, folds_median                                3.0  
                  300 s, val_max                                     3.0

,scenario,config,phase
0,reference,"60 s, folds_median",encoding_test
5,reference,"300 s, folds_median",encoding_test
10,reference,"300 s, val_max",encoding_test
15,20x slower,"60 s, folds_median",encoding_test
20,20x slower,"300 s, folds_median",encoding_test
25,20x slower,"300 s, val_max",encoding_test
30,60x slower,"60 s, folds_median",encoding_test
35,60x slower,"300 s, folds_median",encoding_test
40,60x slower,"300 s, val_max",encoding_test
45,one repeated name,"60 s, folds_median",encoding_test


reference, baseline: detected in 5/5 runs. Median delay: 0.007 s, with 3 tunnel queries before the first alert.
20x slower, baseline: detected in 5/5 runs. Median delay: 0.146 s, with 3 tunnel queries before the first alert.
60x slower, baseline: detected in 5/5 runs. Median delay: 0.439 s, with 3 tunnel queries before the first alert.
Report the time delay alongside the number of queries before the first alert and the number of missed runs.


### Summary: Detection Latency and Early Missed Queries

The experiment evaluates detection latency, the number of tunnel queries preceding the first alert, and the number of completely missed tunnel runs.

All three Isolation Forest configurations detect the tunnel in all five evaluated model runs.

For the reference scenario, the median detection delay is 0.007 seconds, with three tunnel queries occurring before the first alert. The first detected query belongs to the `encoding_test` phase.

Under 20-fold and 60-fold slowdown scenarios, the median detection delay increases to 0.146 and 0.439 seconds, respectively. However, the number of queries before the first alert remains unchanged.

The reported delay represents elapsed capture time between the first tunnel query and the first detected query, not model inference latency or end-to-end security alert delivery time.

Although all runs are detected, the five model runs are not independent attack captures. The results therefore demonstrate early detection for the evaluated tunnel, but do not establish general detection latency across different attacks.

Detection performance should be reported using three complementary measures: elapsed time to first alert, number of tunnel queries before first alert, and number of missed tunnel runs.

## 7. Incident aggregation and false-alert burden

Grouping repeated false alerts from the same client and parent domain should reduce
the number of incidents an analyst has to review. The maximum gap between alerts
can affect that count.

Use the project's grouping rule for each client and parent domain. Start a new
incident when the time since the previous alert exceeds the gap. Test the exact
boundary and check that alerts from different clients or parents stay separate.
Verify that gaps of 30, 60 and 300 seconds leave the number of query alerts unchanged.
Mark an incident as a tunnel incident if it contains at least one true tunnel alert.
Compare false incidents per elapsed test hour on the same test period.

In [ ]:
toy = pd.DataFrame({
    "ts": [0., 60., 121., 60., 60.],
    "client": ["a", "a", "a", "b", "a"],
    "parent": ["x.test", "x.test", "x.test", "x.test", "y.test"],
    "role": ["test"] * 5, "alert": [1] * 5, "y": [0, 1, 0, 0, 0], "score": [.8] * 5,
})
toy_incidents = group_alerts(toy, gap=60)
assert len(toy_incidents) == 4 and toy_incidents.alerts.tolist() == [2, 1, 1, 1]
assert toy_incidents.is_tunnel.sum() == 1

incident_rows = []
for (config, seed), test in scored.items():
    hours = (test.ts.max() - test.ts.min()) / 3600
    fp = int((test.y.eq(0) & test.alert.eq(1)).sum())
    for gap in (30, 60, 300):
        incidents = group_alerts(test, gap=gap)
        assert incidents.alerts.sum() == test.alert.sum()
        counts = incident_metrics(test, gap=gap)
        assert counts["false_incidents"] <= fp
        np.testing.assert_allclose(counts["false_incidents_per_hour"], counts["false_incidents"] / hours)
        incident_rows.append({"config": config, "seed": seed, "gap_s": gap,
                              "fp_queries": fp, "test_hrs": hours, **counts})

incident_results = pd.DataFrame(incident_rows)
incident_summary = incident_results.groupby(["config", "gap_s"], sort=False)[
    ["fp_queries", "test_hrs", "false_incidents", "tunnel_incidents", "false_incidents_per_hour"]
].mean()

display(incident_summary.round(3))

for config in CONFIGS:
    row = incident_summary.loc[(config, 60)]
    print(f"{config}, 60 s gap: an average of {row.fp_queries:.1f} false query alerts and "
          f"{row.false_incidents:.1f} false incidents ({row.false_incidents_per_hour:.2f}/h).")

print("Changing the gap changes incident counts. The underlying query alerts stay fixed.")

fp_queries  test_hrs  false_incidents  \
config              gap_s                                          
60 s, folds_median  30           75.6     0.665              6.4   
                    60           75.6     0.665              3.6   
                    300          75.6     0.665              3.3   
300 s, folds_median 30            9.0     0.665              4.2   
                    60            9.0     0.665              4.1   
                    300           9.0     0.665              3.0   
300 s, val_max      30            3.3     0.665              1.9   
                    60            3.3     0.665              1.9   
                    300           3.3     0.665              1.7   

                           tunnel_incidents  false_incidents_per_hour  
config              gap_s                                              
60 s, folds_median  30                  1.0                     9.624  
                    60                  1.0                     5.413  
                    300                 1.0                     4.962  
300 s, folds_median 30                  1.0                     6.316  
                    60                  1.0                     6.165  
                    300                 1.0                     4.511  
300 s, val_max      30                  1.0                     2.857  
                    60                  1.0                     2.857  
                    300                 1.0                     2.556

60 s, folds_median, 60 s gap: an average of 75.6 false query alerts and 3.6 false incidents (5.41/h).
300 s, folds_median, 60 s gap: an average of 9.0 false query alerts and 4.1 false incidents (6.17/h).
300 s, val_max, 60 s gap: an average of 3.3 false query alerts and 1.9 false incidents (2.86/h).
Changing the gap changes incident counts. The underlying query alerts stay fixed.


### Summary: Incident Aggregation and False Alert Burden

This experiment evaluates how individual DNS-query alerts translate into security incidents under different temporal aggregation rules.

Using a 60-second incident gap, the original detector produces an average of 75.6 false-positive queries, grouped into 3.6 false incidents.

Increasing the observation window to 300 seconds with `folds_median` reduces false-positive queries to 9.0, but slightly increases false incidents to 4.1.

The 300-second `val_max` configuration performs best in this experiment, producing 3.3 false-positive queries and 1.9 false incidents on average.

Compared with the baseline, this represents approximately 95.6% fewer false-positive queries and 47.2% fewer false incidents.

The results demonstrate that query-level precision does not fully represent operational alert burden. Incident counts also depend on the temporal distribution of alerts and the selected aggregation gap.

The reported false incidents per hour describe only the evaluated capture and should not be extrapolated to production environments.

Incident aggregation therefore provides an important complementary evaluation of the detector's practical usability.

The hourly rate uses the full observed test duration, including quiet periods. A false
incident contains only benign alerts. A benign alert in a mixed incident still counts
as a query-level false positive. The incident gap and the feature window are independent.

## 8. Reproducibility and alternative features

The same input, software versions and seed should reproduce scores, thresholds and
alerts. A model with just `uniq_names_60s` and `norepeat_tol` may rank queries as well
as the four-feature model. The project's unique-names rule provides a deterministic
comparison. This feature comparison is exploratory because notebook 04 already used
the same test capture for feature analysis.

Record SHA-256 hashes for the inputs and source modules, together with software
versions and settings. Refit and recalibrate the baseline with seed 0 in the same
environment. Check its test scores and alerts against a direct call to the project API.
Compare the two-feature model on the same test rows and seeds, and report paired
differences. The rule uses the largest count seen during training as its threshold.

In [ ]:
inputs = [_find_raw_dir() / "dns.log", ROOT / "data" / "labels.csv"]
modules = ["prepare", "features", "detector", "evaluation", "incidents", "variants", "baselines"]
files = inputs + [ROOT / "src" / f"{name}.py" for name in modules]

display(pd.Series({path.relative_to(ROOT).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
                   for path in files}, name="SHA-256").to_frame())
packages = ("numpy", "pandas", "scikit-learn", "tldextract", "nbformat", "ipykernel")

display(pd.Series({"python": platform.python_version(), **{name: version(name) for name in packages}},
                  name="version").to_frame())

print(f"Forest settings {DEFAULTS}, main seeds {SEEDS}, stress-test seeds {STRESS_SEEDS}, "
      f"UTC cutoffs {DEFAULT_CUT['first']} and {DEFAULT_CUT['second']}.")

direct, direct_info = run_detector(matrices[60], FEATURES[60], threshold="folds_median", random_state=0)
original = infos[60, 0]
assert original["candidates"] == direct_info["candidates"]
pd.testing.assert_frame_equal(original["folds"], direct_info["folds"], check_exact=True)
pd.testing.assert_frame_equal(
    scored[BASELINE, 0], direct.loc[direct.role.eq("test")][scored[BASELINE, 0].columns],
    check_exact=True,
)

lean_name = "60 s, two features"
lean_features = ["uniq_names_60s", "norepeat_tol"]
lean_rows = []

for seed in SEEDS:
    info = calibrate(matrices[60], lean_features, seed)
    test = score_test(matrices[60], info, "folds_median")
    lean_rows.append({"config": lean_name, "seed": seed, **metrics(test)})
lean = pd.DataFrame(lean_rows).set_index("seed")
base = results.loc[results.config.eq(BASELINE)].set_index("seed")
rule = run_rule(matrices[60], kind="unique_names", window_s=60)
rule_values = pd.Series(metrics(rule.loc[rule.role.eq("test")]), name="unique-names rule")
cols = ["pr_auc", "precision", "recall", "fp", "false_incidents"]

display(pd.DataFrame([base[cols].mean().rename(BASELINE), lean[cols].mean().rename(lean_name),
                      rule_values[cols]]).round(4))
lean_delta = lean[cols] - base[cols]

display(lean_delta.agg(["mean", "std", "min", "max"]).round(4))
print(f"Seed 0 reproduced exactly. Compared with the baseline, the two-feature model changed "
      f"PR AUC by {lean_delta.pr_auc.mean():+.4f} and false query alerts by "
      f"{lean_delta.fp.mean():+.1f} on average.")

,SHA-256
data/raw/Case 1/dns.log,1fb9c5f04960489a0211afef02bc5152691f8a8c2de860...
data/labels.csv,32dd5c180782068dc5bffc798d83cc620c5732ff7af8a5...
src/prepare.py,13013f18fd4d424a8dae5656f30ee843273679eb84757c...
src/features.py,55a6a0d4e00005dad7916a237dab59b0538d49a9d9d2ac...
src/detector.py,6ed605fb166d54e4d85f1ef4ca0ac4433de9ca55a528d0...
src/evaluation.py,8080475f3228d6e28d151257b61b1c29699e3e475d2c19...
src/incidents.py,01ab1a0678ab40f15b2c13fe52615bae0050ad5d6b6a7a...
src/variants.py,aae118b25ba671e7e48c4950bed7f71722d3960267099a...
src/baselines.py,ffe7e1004f7af10d8227dbd552ff5b8699548d67d3fc9f...


,version
python,3.13.5
numpy,2.5.5
pandas,3.1.1
scikit-learn,1.9.1
tldextract,5.4.0
nbformat,5.11.1
ipykernel,7.4.0


Forest settings {'n_estimators': 300, 'max_samples': 'auto', 'max_features': 1.0, 'random_state': 0}, main seeds (0, 1, 2, 3, 4, 5, 6, 7, 8, 9), stress-test seeds (0, 1, 2, 3, 4), UTC cutoffs 2023-01-07 18:04:36 and 2023-01-07 18:30:00.


,pr_auc,precision,recall,fp,false_incidents
"60 s, folds_median",0.9931,0.7442,0.9869,75.6,3.6
"60 s, two features",0.9868,0.7415,0.9851,76.4,2.9
unique-names rule,0.9580,0.7984,0.9279,52.0,1.0


,pr_auc,precision,recall,fp,false_incidents
mean,-0.0064,-0.0027,-0.0018,0.8000,-0.7000
std,0.0020,0.0296,0.0031,11.8771,1.2517
min,-0.0104,-0.0383,-0.0090,-23.0000,-3.0000
max,-0.0044,0.0544,0.0000,14.0000,0.0000


Seed 0 reproduced exactly. Compared with the baseline, the two-feature model changed PR AUC by -0.0064 and false query alerts by +0.8 on average.


The settings, versions and input hashes make the run reproducible. Repeating it with
different seeds measures variation within this capture. Performance on other networks
or captures remains open.

### Summary: Reproducibility and Alternative Feature Sets

This experiment evaluates the reproducibility of the Isolation Forest detector and compares the original four-feature model with a simplified two-feature model and a rule-based alternative.

The notebook records SHA-256 hashes of important data and source files, software versions, model parameters, random seeds, and chronological split boundaries. The original seed-0 experiment was reproduced exactly.

The four-feature Isolation Forest achieves a mean PR AUC of 0.9931 and recall of 0.9869. The two-feature model achieves a slightly lower PR AUC of 0.9868 and recall of 0.9851, without a meaningful reduction in false-positive queries.

The unique-names rule achieves fewer false-positive queries and false incidents, but its recall decreases to 0.9279.

These results support retaining the four-feature Isolation Forest as the main model when high detection recall is prioritized. The simpler alternatives remain useful benchmarks and illustrate the trade-off between detection coverage, complexity, and false-alert burden.

Although seed 0 was reproduced exactly, complete reproducibility across environments still requires verification of the full dependency setup and execution pipeline.

## 9. Alternative anomaly detection algorithms

Compare Isolation Forest, Local Outlier Factor (LOF) and One-Class SVM on the same
300-second features: query length, unique-name count, unique-name ratio and no-repeat
tolerance. All three use the same chronological fit, validation and test rows. The
first 80% of pre-cutoff traffic is used to fit the models and the final 20% to calibrate
each alert threshold. The test labels are used to calculate metrics.

Isolation Forest uses the settings and ten seeds from Section 3. LOF uses 20 neighbours
with novelty detection enabled, and One-Class SVM uses an RBF kernel with
`nu=0.05` and `gamma="scale"`. These settings are fixed for this benchmark.
Each alternative standardizes the four features using statistics from the fit rows.
All three apply `val_max`: an alert requires a score strictly above the largest anomaly
score on that model's benign validation rows.

The table reports average precision (PR AUC), precision, recall, false-positive rate
and false incidents using the shared 60-second incident gap. Isolation Forest is shown
as a mean over seeds with the sample standard deviation underneath. LOF and One-Class
SVM are deterministic for these inputs. This is a retrospective comparison on one
previously examined capture. An independent capture would be needed to confirm the results.

In [12]:
comparison_matrix = matrices[300]
comparison_features = FEATURES[300]
fit_rows, validation_rows = split_validation(
    comparison_matrix.loc[comparison_matrix.split.eq("train")]
)
test_rows = comparison_matrix.loc[comparison_matrix.split.eq("test")]
assert fit_rows.y.eq(0).all() and validation_rows.y.eq(0).all()
assert fit_rows.ts.max() < validation_rows.ts.min() <= validation_rows.ts.max() < test_rows.ts.min()

forest_runs = [scored["300 s, val_max", seed] for seed in SEEDS]
for forest_run in forest_runs:
    pd.testing.assert_frame_equal(
        forest_run[test_rows.columns], test_rows, check_exact=True
    )
forest_metrics = pd.DataFrame([metrics(run) for run in forest_runs])

alternative_models = {
    "Local Outlier Factor": LocalOutlierFactor(n_neighbors=20, novelty=True),
    "One-Class SVM": OneClassSVM(kernel="rbf", nu=0.05, gamma="scale"),
}
alternative_metrics = {}
for name, estimator in alternative_models.items():
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(fit_rows[comparison_features])
    validation_scores = -model.score_samples(validation_rows[comparison_features])
    threshold = validation_scores.max()
    test_scores = -model.score_samples(test_rows[comparison_features])
    assert np.isfinite(validation_scores).all() and np.isfinite(test_scores).all()
    test_alerts = test_rows.assign(
        role="test", score=test_scores, alert=(test_scores > threshold).astype(int)
    )
    alternative_metrics[name] = metrics(test_alerts)

comparison_columns = ["pr_auc", "precision", "recall", "fpr", "false_incidents"]
comparison = pd.DataFrame.from_dict(
    {"Isolation Forest (mean, 10 seeds)": forest_metrics.mean(), **alternative_metrics},
    orient="index",
)[comparison_columns]

display(comparison.round(4))

display(forest_metrics[comparison_columns].std().to_frame().T.rename(
    index={0: "Isolation Forest (seed standard deviation)"}
    ).round(4))

,pr_auc,precision,recall,fpr,false_incidents
"Isolation Forest (mean, 10 seeds)",0.9934,0.9853,0.9865,0.0021,1.9
Local Outlier Factor,0.9638,0.9493,0.9279,0.0068,10.0
One-Class SVM,0.9855,0.6450,0.9820,0.0746,9.0


,pr_auc,precision,recall,fpr,false_incidents
Isolation Forest (seed standard deviation),0.0004,0.0118,0.0,0.0017,0.7379


### Summary: Comparison of Anomaly Detection Algorithms

Three anomaly detection algorithms were evaluated for DNS tunneling detection: Isolation Forest, Local Outlier Factor (LOF), and One-Class SVM.

Isolation Forest achieved the strongest performance across all reported metrics, with a PR AUC of 0.9934, precision of 98.53%, recall of 98.65%, and a false-positive rate of 0.21%.

LOF achieved a precision of 94.93% but a lower recall of 92.79%, indicating that more tunnel queries remained undetected.

One-Class SVM achieved a recall of 98.20%, close to Isolation Forest, but its precision decreased to 64.50%, with a substantially higher false-positive rate of 7.46%.

Isolation Forest also produced the lowest operational alert burden, averaging 1.9 false incidents compared with 10.0 for LOF and 9.0 for One-Class SVM.

The Isolation Forest results were stable across ten random seeds, with a PR AUC standard deviation of 0.0004.

These findings support selecting Isolation Forest as the preferred anomaly detection algorithm for the evaluated DNS traffic. However, the conclusions are limited to the available dataset and experimental setup. Generalization to other network environments and tunneling techniques remains unverified.